# Rag Assignment
Authors: Gianluca Amato and David Farrugia

## 1. Setup

### 1.1 Imports

In [12]:
import time
import requests
import pandas as pd
from pathlib import Path
from IPython.display import display
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\david\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\david\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

### 1.2 Path Validation

This section ensures that the required directory structure exists before any data processing begins. Output and temporary dataset directories are created if they do not already exist, which prevents runtime errors during file writing.

A helper function is defined to validate CSV file paths. It checks:

- whether the file exists,
- whether the path refers to a file (and not a directory),
- whether the file has a `.csv` extension.

This validation step guarantees that the Kaggle Financial Sentiment Analysis dataset is correctly located before preprocessing starts.

In [13]:
# Create Dataset Directories if they don't exist
DATA_OUTPUT_DIR = Path("./Datasets/Outputs")
DATA_TEMP_DIR = Path("./Datasets/TempDatasets")

for directory in [DATA_OUTPUT_DIR, DATA_TEMP_DIR]:
        directory.mkdir(parents=True, exist_ok=True)

# Paths
DATA_INPUT_DIR = Path("./Datasets/Inputs")
FSA_PATH = DATA_INPUT_DIR / "FSA_data.csv"

def validate_csv_path(path: Path, label: str):
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path.resolve()}")
    if not path.is_file():
        raise ValueError(f"{label} exists but is not a file: {path.resolve()}")
    if path.suffix.lower() != ".csv":
        raise ValueError(f"{label} is not a CSV file: {path.resolve()}")
    print(f"[OK] {label}: {path.resolve()}")
    
validate_csv_path(FSA_PATH, "FSA Data")

[OK] FSA Data: C:\Users\david\Desktop\GitHub\ICS5111-MiningLargeScaleData_Project\Datasets\Inputs\FSA_data.csv


## 2. Data Processing

In this section, a stop-word set is initialised using NLTK’s English stop words. A reusable preprocessing function is then implemented to:
- lowercase text,
- tokenize sentences,
- remove stop words,
- remove non-alphabetic tokens,
- optionally retain year tokens (four-digit numbers).

Additionally, this function produces a sparse textual representation, which is suitable for keyword-based retrieval techniques such as TF-IDF or BM25.

In [14]:
# Get English stop words set
stop_words = set(stopwords.words('english'))
    
# Function to preprocess text for sparse representation
def tokenize_and_filter(text, keep_years=False):
    # Lowercase, tokenize, remove stopwords and non-alphabetic tokens
    tokens = word_tokenize(text.lower())
    
    filtered_tokens = []
    
    # Filter tokens
    for word in tokens:
        # keep alphabetic words
        if word.isalpha() and word not in stop_words:
            filtered_tokens.append(word)
            
            # optionally keep years (4-digit numbers)
        elif keep_years == True and word.isdigit() and len(word) == 4:
            filtered_tokens.append(word)
        
    # Join tokens back into a single string
    return " ".join(filtered_tokens)

#### 2.1 [Kaggle Financial Sentiment Analysis Dataset](https://www.kaggle.com/sbhatti/financial-sentiment-analysis)

In this step, the Kaggle dataset is loaded from disk and prepared for retrieval.

Each sentence is:
- assigned a `source` label (`kaggle`) to track provenance,
- converted into a sparse representation using the preprocessing function,
- renamed to use a consistent `text` column for dense retrieval.

The dataset is then reordered for clarity and saved to disk as a processed corpus. This processed version is later used directly in the unified RAG dataset.

In [15]:
# Load dataset from disk
dataset = pd.read_csv(FSA_PATH)

# Apply preprocessing to dataset for each sentence and reorder columns for better readability
dataset["source"] = "kaggle"
dataset["processed_text"] = dataset["Sentence"].apply(tokenize_and_filter)

dataset = dataset.rename(columns={"Sentence": "text", "Sentiment": "sentiment"})
dataset = dataset[["source", "text", "processed_text", "sentiment"]]

print("Kaggle FSA Processed Data Sample:")
display(dataset.head(10))

# Save processed dataset to disk
KAGGLE_OUTPUT_PATH = DATA_TEMP_DIR / "kaggle_fsa_dataset.csv"
dataset.to_csv(KAGGLE_OUTPUT_PATH, index=False)

Kaggle FSA Processed Data Sample:


,source,text,processed_text,sentiment
0,kaggle,The GeoSolutions technology will leverage Bene...,geosolutions technology leverage benefon gps s...,positive
1,kaggle,"$ESI on lows, down $1.50 to $2.50 BK a real po...",esi lows bk real possibility,negative
2,kaggle,"For the last quarter of 2010 , Componenta 's n...",last quarter componenta net sales doubled peri...,positive
3,kaggle,According to the Finnish-Russian Chamber of Co...,according chamber commerce major construction ...,neutral
4,kaggle,The Swedish buyout firm has sold its remaining...,swedish buyout firm sold remaining percent sta...,neutral
5,kaggle,$SPY wouldn't be surprised to see a green close,spy would surprised see green close,positive
6,kaggle,Shell's $70 Billion BG Deal Meets Shareholder ...,shell billion bg deal meets shareholder skepti...,negative
7,kaggle,SSH COMMUNICATIONS SECURITY CORP STOCK EXCHANG...,ssh communications security corp stock exchang...,negative
8,kaggle,Kone 's net sales rose by some 14 % year-on-ye...,kone net sales rose first nine months,positive
9,kaggle,The Stockmann department store will have a tot...,stockmann department store total floor space s...,neutral


### 2.2 [World Bank Open Data](https://data.worldbank.org/)

This section retrieves structured macroeconomic indicators from the World Bank Open Data API.

For each selected indicator and country:
- data is fetched year by year,
- API rate limits are respected using a delay between requests,
- results are stored in a DataFrame.

Each row is labelled with a `source` identifier (`wb_open_data`) to distinguish it from the kaggle sources. The resulting dataset contains structured numerical values along with associated metadata such as country, year, and indicator name.

In [16]:
WB_API_URL = "https://api.worldbank.org/v2/country/{country}/indicator/{indicator}"

# Function to fetch data from World Bank API
def fetch_indicator_data(indicator, countries, start_year, end_year, per_page=2000, rate_limit_delay=0.2):
    '''
    This fuction fetches data from World Bank API for given indicator and countries within specified year range.
    
    Parameters:
    - indicator (str): World Bank indicator code
    - contries (list): List of country codes
    - start_year (int): Start year for data
    - end_year (int): End year for data
    - per_page (int): Number of records per page
    - rate_limit_delay (float): Delay between requests to avoid rate limiting
    
    Returns:
    - DataFrame with columns ['country', 'year', 'value']
    '''
    all_records = []
    
    # Loop through countries one by one
    for country in countries:
        # Construct API URL
        url = WB_API_URL.format(
            country=country,
            indicator=indicator,
        )
        
        # Set query parameters
        params = {
            "date": f"{start_year}:{end_year}",
            "format": "json",
            "per_page": per_page,
        }
        
        # Make API request
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status() # Raise error for bad responses
        data = response.json()
    
        if len(data) < 2 or not data[1]:
            # No returned data for this country, skip
            time.sleep(rate_limit_delay)
            continue
        
        # Append records for this country
        for entry in data[1]:
            all_records.append({
                "country": entry["country"]["value"],
                "year": entry["date"],
                "value": entry["value"]
            })
    
        # Wait before next request so we don't spam the API
        time.sleep(rate_limit_delay)
    
    return pd.DataFrame(all_records)

After fetching, the World Bank dataset is inspected to verify correctness and completeness. A preview of the data confirms that the expected indicators and years are present.

The dataset is then saved to disk as a CSV file, allowing reproducibility and separation between data acquisition and later processing steps.

In [17]:
# Congifure parameters for data fetching
COUNTRIES = {
    "MLT": "Malta",
    "ITA": "Italy",
    "DEU": "Germany",
    "FRA": "France",
    "ESP": "Spain",
    "PRT": "Portugal",
    "IRL": "Ireland",
    "NLD": "Netherlands",
}

INDICATORS = {
    "NY.GDP.MKTP.KD.ZG": "GDP growth (%)",
    "FP.CPI.TOTL.ZG": "Inflation rate (annual %)",
    "GC.DOD.TOTL.GD.ZS": "Government debt (% of GDP)",
    "SL.EMP.TOTL.ZS": "Employment rate (% of total labor force)",
    "SL.UEM.TOTL.ZS": "Unemployment rate (% of total labor force)"
}

START_YEAR = 2000
END_YEAR = 2025

# Fetch data
wb_data_df = []

for indicator_code, indicator_name in INDICATORS.items():
    df = fetch_indicator_data(
        indicator=indicator_code,
        countries=list(COUNTRIES.keys()),
        start_year=START_YEAR,
        end_year=END_YEAR
        # per_page = 2000,          # Parameters already set to default values in function
        # rate_limit_delay = 0.2    # Parameters already set to default values in function
    )

    df["indicator_name"] = indicator_name
    wb_data_df.append(df)

wb_data_df = pd.concat(wb_data_df, ignore_index=True)
wb_data_df["source"] = "wb_open_data"

# Reorder columns for better readability
wb_data_df = wb_data_df[["source", "country", "year", "indicator_name", "value"]]

print("World Bank Economic Indicators Data Sample:")
display(wb_data_df.head(10))

# Save fetched data to CSV
WB_OUTPUT_PATH = DATA_TEMP_DIR / "wb_economic_indicators_dataset.csv"
wb_data_df.to_csv(WB_OUTPUT_PATH, index=False)

World Bank Economic Indicators Data Sample:


,source,country,year,indicator_name,value
0,wb_open_data,Malta,2024,GDP growth (%),6.799903
1,wb_open_data,Malta,2023,GDP growth (%),10.621519
2,wb_open_data,Malta,2022,GDP growth (%),2.483100
3,wb_open_data,Malta,2021,GDP growth (%),13.411704
4,wb_open_data,Malta,2020,GDP growth (%),-3.457283
5,wb_open_data,Malta,2019,GDP growth (%),4.085056
6,wb_open_data,Malta,2018,GDP growth (%),7.189215
7,wb_open_data,Malta,2017,GDP growth (%),12.971342
8,wb_open_data,Malta,2016,GDP growth (%),4.078004
9,wb_open_data,Malta,2015,GDP growth (%),9.620703


### 2.3 Combinging Both Datasets

In [18]:
validate_csv_path(KAGGLE_OUTPUT_PATH, "Kaggle FSA Processed Data")
validate_csv_path(WB_OUTPUT_PATH, "World Bank Economic Indicators Data")

[OK] Kaggle FSA Processed Data: C:\Users\david\Desktop\GitHub\ICS5111-MiningLargeScaleData_Project\Datasets\TempDatasets\kaggle_fsa_dataset.csv
[OK] World Bank Economic Indicators Data: C:\Users\david\Desktop\GitHub\ICS5111-MiningLargeScaleData_Project\Datasets\TempDatasets\wb_economic_indicators_dataset.csv


In [19]:
# read csv files back to verify
kaggle_df = pd.read_csv(KAGGLE_OUTPUT_PATH)
wb_df = pd.read_csv(WB_OUTPUT_PATH)

print("Kaggle FSA Processed Data Sample:")
display(kaggle_df.head())
print("World Bank Economic Indicators Data Sample:")
display(wb_df.head())

Kaggle FSA Processed Data Sample:


,source,text,processed_text,sentiment
0,kaggle,The GeoSolutions technology will leverage Bene...,geosolutions technology leverage benefon gps s...,positive
1,kaggle,"$ESI on lows, down $1.50 to $2.50 BK a real po...",esi lows bk real possibility,negative
2,kaggle,"For the last quarter of 2010 , Componenta 's n...",last quarter componenta net sales doubled peri...,positive
3,kaggle,According to the Finnish-Russian Chamber of Co...,according chamber commerce major construction ...,neutral
4,kaggle,The Swedish buyout firm has sold its remaining...,swedish buyout firm sold remaining percent sta...,neutral


World Bank Economic Indicators Data Sample:


,source,country,year,indicator_name,value
0,wb_open_data,Malta,2024,GDP growth (%),6.799903
1,wb_open_data,Malta,2023,GDP growth (%),10.621519
2,wb_open_data,Malta,2022,GDP growth (%),2.483100
3,wb_open_data,Malta,2021,GDP growth (%),13.411704
4,wb_open_data,Malta,2020,GDP growth (%),-3.457283


Since retrieval models operate on text rather than raw numerical tables, each World Bank data row is converted into a natural-language sentence.

For example:
> **“In 2024, Malta’s GDP growth (%) was 6.79.”**

This transformation allows for structured indicators to be indexed and retrieved in the same way as unstructured text documents.

In [20]:
# Create natural language representation for each World Bank data row
def wb_row_to_text(row):
    return f"In {row['year']}, {row['country']}'s {row['indicator_name']} was {row['value']:.6f}."

wb_data_df["text"] = wb_data_df.apply(wb_row_to_text, axis=1)
display(wb_data_df.head())

,source,country,year,indicator_name,value,text
0,wb_open_data,Malta,2024,GDP growth (%),6.799903,"In 2024, Malta's GDP growth (%) was 6.799903."
1,wb_open_data,Malta,2023,GDP growth (%),10.621519,"In 2023, Malta's GDP growth (%) was 10.621519."
2,wb_open_data,Malta,2022,GDP growth (%),2.483100,"In 2022, Malta's GDP growth (%) was 2.483100."
3,wb_open_data,Malta,2021,GDP growth (%),13.411704,"In 2021, Malta's GDP growth (%) was 13.411704."
4,wb_open_data,Malta,2020,GDP growth (%),-3.457283,"In 2020, Malta's GDP growth (%) was -3.457283."


The generated natural language sentences are further processed using the same preprocessing function used for the Kaggle dataset.

In this case, year tokens are retained to support time-specific queries (e.g., “GDP growth in 2020”). This produces a sparse representation suitable for keyword-based retrieval while preventing the loss of temporal information.

In [21]:
wb_data_df["processed_text"] = wb_data_df["text"].apply(tokenize_and_filter, keep_years=True)
display(wb_data_df.head())


,source,country,year,indicator_name,value,text,processed_text
0,wb_open_data,Malta,2024,GDP growth (%),6.799903,"In 2024, Malta's GDP growth (%) was 6.799903.",2024 malta gdp growth
1,wb_open_data,Malta,2023,GDP growth (%),10.621519,"In 2023, Malta's GDP growth (%) was 10.621519.",2023 malta gdp growth
2,wb_open_data,Malta,2022,GDP growth (%),2.483100,"In 2022, Malta's GDP growth (%) was 2.483100.",2022 malta gdp growth
3,wb_open_data,Malta,2021,GDP growth (%),13.411704,"In 2021, Malta's GDP growth (%) was 13.411704.",2021 malta gdp growth
4,wb_open_data,Malta,2020,GDP growth (%),-3.457283,"In 2020, Malta's GDP growth (%) was -3.457283.",2020 malta gdp growth


Before merging, empty metadata columns are added to the Kaggle dataset to ensure schema consistency between both sources.

Both datasets are then reordered to share the same column structure:

- source
- text
- processed_text
- country
- year
- indicator_name
- value

They are combined using a concatenation operation to form a single unified dataset.

This final `.csv` file represents the `complete RAG dataset` for this project, contanining a combination of data from two sources

In [22]:
# In order to keep metadata consistent across both datasets, we add empty columns to the Kaggle dataset
kaggle_df["country"] = None
kaggle_df["year"] = None
kaggle_df["indicator_name"] = None
kaggle_df["value"] = None

# Final reordering of columns
kaggle_df = kaggle_df[
    ["source", "text", "processed_text", "country", "year", "indicator_name", "value"]
]

wb_data_df = wb_data_df[
    ["source", "text", "processed_text", "country", "year", "indicator_name", "value"]
]

# Combine both datasets into final RAG dataset
final_rag_dataset = pd.concat([kaggle_df, wb_data_df], ignore_index=True)

print("Final RAG Dataset Sample:")
display(final_rag_dataset.head())

# Save final RAG dataset to CSV
RAG_OUTPUT_PATH = DATA_OUTPUT_DIR / "final_rag_dataset.csv"
final_rag_dataset.to_csv(RAG_OUTPUT_PATH, index=False)

Final RAG Dataset Sample:


C:\Users\david\AppData\Local\Temp\ipykernel_1324\3449096297.py:17: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_rag_dataset = pd.concat([kaggle_df, wb_data_df], ignore_index=True)


,source,text,processed_text,country,year,indicator_name,value
0,kaggle,The GeoSolutions technology will leverage Bene...,geosolutions technology leverage benefon gps s...,None,None,None,NaN
1,kaggle,"$ESI on lows, down $1.50 to $2.50 BK a real po...",esi lows bk real possibility,None,None,None,NaN
2,kaggle,"For the last quarter of 2010 , Componenta 's n...",last quarter componenta net sales doubled peri...,None,None,None,NaN
3,kaggle,According to the Finnish-Russian Chamber of Co...,according chamber commerce major construction ...,None,None,None,NaN
4,kaggle,The Swedish buyout firm has sold its remaining...,swedish buyout firm sold remaining percent sta...,None,None,None,NaN
